In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent

import os
from dotenv import load_dotenv
load_dotenv()

C:\Users\vivek\AppData\Local\Temp\ipykernel_20052\3500639950.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


True

In [2]:
# db = SQLDatabase.from_uri("sqlite:///data/employees_db-full-1.0.6.db")
db = SQLDatabase.from_uri("sqlite:///../data/employees_db-full-1.0.6.db")

try:
    tables = db.get_usable_table_names()
    print("Successfully connected to the database.")
    print(f"Number of tables found: {len(tables)}, Tables: {', '.join(tables)}")
except Exception as e:
    print("Failed to connect to the database.")
    print(f"Error: {e}")

schema = db.get_table_info()


Successfully connected to the database.
Number of tables found: 6, Tables: departments, dept_emp, dept_manager, employees, salaries, titles


In [3]:
llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        project="ai-framework-projects",
        temperature=0
        )

In [4]:
@tool
def get_database_schema(table_name: str = None):
    """
    Get the database schema for generating the SQL queries.
    Use this first to understand the table structure and relationships before generating SQL queries.
    """
    if table_name:
        try:
            tables = db.get_usable_table_names()
            if table_name.lower() in [t.lower() for t in tables]:
                result = db.get_table_info([table_name])
                print("Retrived the schema for the table: ", table_name)
                return result
            else:
                return f"Table '{table_name}' does not exist in the database. Available tables: {', '.join(tables)}"
        except Exception as e:
            return f"Error getting the schema for the table {table_name}: {e}"
    else:
        print("Retrived Full database schema.")
        return db.get_table_info()

In [5]:
get_database_schema.invoke({'table_name': 'dept_emp'})

Retrived the schema for the table:  dept_emp


'\nCREATE TABLE dept_emp (\n\temp_no INTEGER NOT NULL, \n\tdept_no CHAR(4) NOT NULL, \n\tfrom_date DATE NOT NULL, \n\tto_date DATE NOT NULL, \n\tPRIMARY KEY (emp_no, dept_no), \n\tFOREIGN KEY(dept_no) REFERENCES departments (dept_no), \n\tFOREIGN KEY(emp_no) REFERENCES employees (emp_no)\n)\n\n/*\n3 rows from dept_emp table:\nemp_no\tdept_no\tfrom_date\tto_date\n10001\td005\t1986-06-26\t9999-01-01\n10002\td007\t1996-08-03\t9999-01-01\n10003\td004\t1995-12-03\t9999-01-01\n*/'

In [6]:
@tool
def generate_sql_query(question: str, schema_info: str = None):
    """
    Generate a SQL SELECT query from a natural language question using database schema.
    Always use this after getting schema information...
    """
    print(f"Generating SQL Query for {question[:100]}...")

    # Use provided schema or get full schema
    schema_to_use = schema_info if schema_info else db.get_table_info()

    prompt = f"""
                Based on this database schema:
                {schema_to_use}

                Generate a SQL query to answer this question: {question}

                Rules:
                - Use only SELECT statements
                - Include only existing columns and tables
                - Add appropriate WHERE, GROUP BY, ORDER BY clauses as needed
                - Limit results to 10 rows unless specified otherwise
                - User proper SQL syntax for SQLlite

                Return only the SQL query, nothing else.
             """
    try:
        response = llm.invoke(prompt)
        query = response.content.strip()
        print(f"Generated SQL query")
        return query
    except Exception as e:
        return f"Error generating query: {e}"

In [7]:
result = generate_sql_query.invoke({"question": "What is maximum salary in employees"})

Generating SQL Query for What is maximum salary in employees...
Generated SQL query


In [8]:
print(result)

```sql
SELECT MAX(salary)
FROM salaries;
```


In [9]:
import re

@tool
def validate_sql_query(query: str) -> str:
    """Validate SQL Query for saftey and syntax before execution.
    Returns 'Valid: <query>' if safe or 'Error: <message>' if unsafe.
    """
    print(f" Validating SQL: {query[:100]}")

    clean_query = query.strip()

    # Remove SQL code block markers if present
    clean_query = re.sub(r'```sql\s*', '', clean_query, flags=re.IGNORECASE)
    clean_query = re.sub(r'```\s*', '', clean_query)
    clean_query = clean_query.strip().rstrip(";")

    # Check 1: Must be a SELECT statement
    if not clean_query.lower().startswith("select"):
        return "Error: Only SELECT statements are allowed"
    
    # Check 2: Block dangerous SQL keywords
    dangerous_keywords = ['INSERT', 'UPDATE' 'DELETE', 'ALTER', 'DROP', 'CREATE', 'TRUNCATE']
    query_upper = clean_query.upper()

    for keyword in dangerous_keywords:
        if keyword in query_upper:
            return f"Error: {keyword} operations are not allowed"
    
    print("Query validation passed")
    return f"Valid: {clean_query}"


In [11]:
result = validate_sql_query.invoke({"query": "SELECT MAX(salary) FROM salaries;"})
print(result)

result = validate_sql_query.invoke({"query": "Update MAX(salary) FROM salaries;"})
print(result)

 Validating SQL: SELECT MAX(salary) FROM salaries;
Query validation passed
Valid: SELECT MAX(salary) FROM salaries
 Validating SQL: Update MAX(salary) FROM salaries;
Error: Only SELECT statements are allowed


In [ ]:
@tool
def execute_sql_query(query: str) -> str:
    """
    Excute a vaildated SQL Query and return the results.
    Only use this after validating the query for safety.
    """
    print(f"Executing the SQL Query {query[:100]}")

    try:
        clean_query = query.strip()
        if clean_query.startswith("Valid: "):
            clean_query = clean_query[7:]
        
        clean_query = re.sub(r'```sql\s*', '', clean_query, flags=re.IGNORECASE)
        clean_query = re.sub(r'```\s*', '', clean_query)
        clean_query = clean_query.strip().rstrip(";")

        result = db.run(clean_query)
        print("Query executed sucessfully")

        if result:
            return f"Query results:\n{result}"
        else:
            return "Query exectued sucessfully but returned no results."

    except Exception as e:
        error_msg = f"Exectuion Error: {str(e)}"
        print(f"{error_msg}")
        return error_msg



_IncompleteInputError: incomplete input (4039884348.py, line 24)